# Notebook 23 — Hybrid GRU+FC Optuna Re-Tuning (Phase 4 Tier 3A)

**Goal:** Tune the Hybrid GRU+FC architecture with Optuna. NB21 used first-guess hyperparameters.
LGBM re-tuning (NB12) gave +0.0046 CV. Expected +0.003–0.007 here → proj LB 0.939–0.944.

## Search space

| Hyperparameter | NB21 default | Search range |
|---|---|---|
| `window_size` | 10 | categorical [5, 10, 15, 20] |
| `gru_hidden` | 128 | categorical [64, 128, 256] |
| `fc_hidden` | 64 | categorical [32, 64, 128] |
| `gru_dropout` | 0.15 | float [0.05, 0.30] |
| `head_dropout1` | 0.25 | float [0.10, 0.40] |
| `head_dropout2` | 0.10 | float [0.05, 0.20] |
| `lr` | 5e-4 | log-float [1e-4, 1e-3] |
| `weight_decay` | 1e-4 | log-float [1e-5, 1e-3] |
| `use_attention` | False | categorical [True, False] |

## Strategy
- **Optuna:** 50 trials, evaluated on folds 0+3 (2-fold mean AUC).
  Kaggle T4×2: 2 parallel workers via `multiprocessing` + SQLite-backed study (~5h).
  Local CPU: single-process fallback.
- **Final training:** Full 5-fold with best params + SWA (10 extra epochs after convergence).
- **Submit solo** — use +0.0177 boost for LB projection. Optionally rank-avg with NB21 seeds.

**Inputs:** `train_features_v2.parquet`, `test_features_v2.parquet`, `fold_assignments.parquet`
**Outputs:** `oof_predictions_nb23.parquet`, `test_predictions_nb23.parquet`, `models/hybrid_nb23_fold{0-4}.pt`

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
import optuna
import pickle, time, warnings
from pathlib import Path
from multiprocessing import Process
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
if Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob('train_features_v2.parquet'))
    if not candidates:
        raise FileNotFoundError('Upload processed parquets as a Kaggle dataset.')
    processed_dir = candidates[0].parent
    models_dir    = Path('/kaggle/working/models')
    working_dir   = Path('/kaggle/working')
else:
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    project_root  = cwd
    processed_dir = project_root / 'data' / 'processed'
    models_dir    = project_root / 'models'
    working_dir   = project_root / 'models'

models_dir.mkdir(exist_ok=True)
print(f'processed_dir: {processed_dir}')
print(f'models_dir   : {models_dir}')

train    = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test     = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Folds: {folds_df.fold.value_counts().sort_index().to_dict()}')

In [ ]:
SEQ_COLS = [
    'LapTime (s)', 'TyreLife', 'Cumulative_Degradation_winsorized',
    'LapTime_Delta', 'Position',
]

STRAT_COLS = [
    'Stint', 'RaceProgress', 'laps_remaining', 'compound_ordinal', 'is_wet_tyre',
    'prime_pit_window', 'laps_to_driver_end', 'abs_position_change',
    'pos_change_rolling_std_3', 'PitStop_lag1',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
assert len(STRAT_COLS) == 13

CAT_COLS         = ['Driver', 'Race', 'Compound', 'Year']
POS_WEIGHT       = 4.03
N_FOLDS          = 5
TUNING_FOLDS     = [0, 3]
WINDOW_CANDIDATES = [5, 10, 15, 20]

N_TRIALS_PER_WORKER = 25           # 2 workers x 25 = 50 total
STUDY_NAME          = 'hybrid_optuna_nb23'
STORAGE_URL         = f'sqlite:///{working_dir}/hybrid_optuna_nb23.db'

TUNE_MAX_EPOCHS  = 50
TUNE_PATIENCE    = 7
TUNE_BATCH_SIZE  = 2048

FINAL_MAX_EPOCHS = 80
FINAL_PATIENCE   = 10
FINAL_BATCH_SIZE = 2048
SWA_EPOCHS       = 10
SWA_LR           = 5e-5

print(f'Tuning folds     : {TUNING_FOLDS}')
print(f'Optuna trials    : {N_TRIALS_PER_WORKER * 2} total')
print(f'Window candidates: {WINDOW_CANDIDATES}')

In [ ]:
combined_cats = pd.concat([train[CAT_COLS], test[CAT_COLS]], ignore_index=True)
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cats[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} classes')

for col in CAT_COLS:
    le = label_encoders[col]
    train[col + '_idx'] = le.transform(train[col].astype(str))
    test[col  + '_idx'] = le.transform(test[col].astype(str))

EMB_DIMS = {
    'Driver':   (len(label_encoders['Driver'].classes_)   + 1, 32),
    'Race':     (len(label_encoders['Race'].classes_)     + 1, 8),
    'Compound': (len(label_encoders['Compound'].classes_) + 1, 3),
    'Year':     (len(label_encoders['Year'].classes_)     + 1, 2),
}
EMB_TOTAL = sum(v[1] for v in EMB_DIMS.values())  # 45
print(f'Embedding total dim: {EMB_TOTAL}')

In [ ]:
def build_windows(combined_df, window_size, seq_scaler):
    N    = len(combined_df)
    N_SF = len(SEQ_COLS)
    scaled = seq_scaler.transform(combined_df[SEQ_COLS].values).astype(np.float32)

    windows = np.zeros((N, window_size, N_SF), dtype=np.float32)
    masks   = np.zeros((N, window_size),       dtype=bool)

    GROUP_COLS = ['Race', 'Year', 'Driver']
    for _, grp_idx in tqdm(
            combined_df.groupby(GROUP_COLS, sort=False).groups.items(),
            total=combined_df[GROUP_COLS].drop_duplicates().shape[0],
            desc=f'W={window_size}', leave=False):
        idxs  = grp_idx.values
        for i in range(len(idxs)):
            hist_len = min(i, window_size)
            if hist_len > 0:
                windows[idxs[i], window_size - hist_len:] = scaled[idxs[i - hist_len : i]]
                masks[idxs[i],   window_size - hist_len:] = True

    return windows, masks

In [ ]:
# Build combined df once; sort for temporal ordering
idx_cols     = ['id', 'Race', 'Year', 'Driver', 'LapNumber']
cat_idx_cols = [c + '_idx' for c in CAT_COLS]
all_sel = list(dict.fromkeys(idx_cols + SEQ_COLS + STRAT_COLS + cat_idx_cols))

combined_df = pd.concat([
    train[all_sel + ['PitNextLap']].assign(is_train=True),
    test[ all_sel].assign(is_train=False, PitNextLap=-1),
], ignore_index=True)
combined_df = combined_df.sort_values(['Race', 'Year', 'Driver', 'LapNumber']).reset_index(drop=True)

seq_scaler = StandardScaler()
seq_scaler.fit(train[SEQ_COLS].values)

PREBUILT_WINDOWS = {}
PREBUILT_MASKS   = {}
t0 = time.time()
for ws in WINDOW_CANDIDATES:
    print(f'Building W={ws}...', end=' ', flush=True)
    w, m = build_windows(combined_df, ws, seq_scaler)
    PREBUILT_WINDOWS[ws] = w
    PREBUILT_MASKS[ws]   = m
    print(f'done  {w.shape}')
print(f'Total: {time.time()-t0:.1f}s')

In [ ]:
tr_mask = combined_df['is_train'].values
te_mask = ~tr_mask

TRAIN_WINDOWS = {ws: PREBUILT_WINDOWS[ws][tr_mask] for ws in WINDOW_CANDIDATES}
TRAIN_MASKS   = {ws: PREBUILT_MASKS[ws][tr_mask]   for ws in WINDOW_CANDIDATES}
TEST_WINDOWS  = {ws: PREBUILT_WINDOWS[ws][te_mask]  for ws in WINDOW_CANDIDATES}
TEST_MASKS    = {ws: PREBUILT_MASKS[ws][te_mask]    for ws in WINDOW_CANDIDATES}

TRAIN_STRAT_RAW = combined_df.loc[tr_mask, STRAT_COLS].values.astype(np.float32)
TEST_STRAT_RAW  = combined_df.loc[te_mask, STRAT_COLS].values.astype(np.float32)
TRAIN_TARGETS   = combined_df.loc[tr_mask, 'PitNextLap'].values.astype(np.float32)
TRAIN_IDS       = combined_df.loc[tr_mask, 'id'].values
TEST_IDS        = combined_df.loc[te_mask, 'id'].values

fold_lookup  = folds_df.set_index('id')['fold']
TRAIN_FOLDS  = fold_lookup.loc[TRAIN_IDS].values
TRAIN_CAT    = {c: combined_df.loc[tr_mask, c+'_idx'].values for c in CAT_COLS}
TEST_CAT     = {c: combined_df.loc[te_mask, c+'_idx'].values for c in CAT_COLS}

del PREBUILT_WINDOWS, PREBUILT_MASKS, combined_df

print(f'Train windows W=10: {TRAIN_WINDOWS[10].shape}  strat: {TRAIN_STRAT_RAW.shape}')
print(f'Test  windows W=10: {TEST_WINDOWS[10].shape}')
print(f'Fold counts: {pd.Series(TRAIN_FOLDS).value_counts().sort_index().to_dict()}')

# Save arrays to disk for spawn-safe subprocess workers
print('Saving arrays to disk for workers...', end=' ', flush=True)
for ws in WINDOW_CANDIDATES:
    np.save(working_dir / f'train_win_{ws}.npy',  TRAIN_WINDOWS[ws])
    np.save(working_dir / f'train_mask_{ws}.npy', TRAIN_MASKS[ws])
    np.save(working_dir / f'test_win_{ws}.npy',   TEST_WINDOWS[ws])
    np.save(working_dir / f'test_mask_{ws}.npy',  TEST_MASKS[ws])
np.save(working_dir / 'train_strat.npy',   TRAIN_STRAT_RAW)
np.save(working_dir / 'test_strat.npy',    TEST_STRAT_RAW)
np.save(working_dir / 'train_targets.npy', TRAIN_TARGETS)
np.save(working_dir / 'train_folds.npy',   TRAIN_FOLDS.astype(np.int32))
import pickle
with open(working_dir / 'train_cat.pkl', 'wb') as f: pickle.dump(TRAIN_CAT, f)
with open(working_dir / 'test_cat.pkl',  'wb') as f: pickle.dump(TEST_CAT, f)
with open(working_dir / 'emb_dims.pkl',  'wb') as f: pickle.dump(EMB_DIMS, f)
print('done')

## Model — Generalized HybridGRUFC

Same dual-branch design as NB21 but with tunable `gru_hidden`, `fc_hidden`, and an optional temporal attention pooling layer that replaces `h_n[-1]` with a learned weighted sum over all GRU timesteps.

In [ ]:
class HybridGRUFC(nn.Module):
    """Dual-branch GRU+FC Hybrid. All key sizes are tunable for Optuna."""

    def __init__(self, n_strat=13, n_seq=5, gru_hidden=128, fc_hidden=64,
                 n_layers=2, gru_drop=0.15, head_drop1=0.25, head_drop2=0.10,
                 use_attention=False, emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS

        self.use_attention = use_attention

        # Entity embeddings (shared)
        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])

        # GRU temporal branch
        self.gru = nn.GRU(
            input_size=n_seq,
            hidden_size=gru_hidden,
            num_layers=n_layers,
            batch_first=True,
            dropout=gru_drop if n_layers > 1 else 0.0,
        )

        # Temporal attention pooling (optional)
        if use_attention:
            self.attn_proj = nn.Linear(gru_hidden, 1)

        # FC strategic branch
        self.strat_fc = nn.Sequential(
            nn.Linear(n_strat, fc_hidden),
            nn.BatchNorm1d(fc_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(fc_hidden, fc_hidden),
            nn.BatchNorm1d(fc_hidden),
            nn.ReLU(),
        )

        # Merge classification head
        emb_total = sum(v[1] for v in emb_dims.values())
        merge_dim = gru_hidden + fc_hidden + emb_total
        self.head = nn.Sequential(
            nn.Linear(merge_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(head_drop1),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(head_drop2),
            nn.Linear(64, 1),
        )

    def forward(self, window, mask, strat, driver, race, compound, year):
        emb = torch.cat([
            self.driver_emb(driver),
            self.race_emb(race),
            self.compound_emb(compound),
            self.year_emb(year),
        ], dim=1)  # (B, 45)

        # GRU branch
        seq_len = mask.sum(dim=1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False
        )

        if self.use_attention:
            gru_out_packed, _ = self.gru(packed)
            gru_out, _ = nn.utils.rnn.pad_packed_sequence(
                gru_out_packed, batch_first=True, total_length=window.size(1)
            )  # (B, W, gru_hidden)
            attn_scores  = self.attn_proj(gru_out)          # (B, W, 1)
            attn_scores  = attn_scores.masked_fill(~mask.unsqueeze(-1), -1e9)
            attn_weights = torch.softmax(attn_scores, dim=1)  # (B, W, 1)
            gru_feat     = (attn_weights * gru_out).sum(dim=1)  # (B, gru_hidden)
        else:
            _, h_n   = self.gru(packed)
            gru_feat = h_n[-1]  # (B, gru_hidden)

        strat_feat = self.strat_fc(strat)  # (B, fc_hidden)
        merged     = torch.cat([gru_feat, strat_feat, emb], dim=1)
        return self.head(merged).squeeze(1)


# Sanity-check default config (matches NB21)
_m = HybridGRUFC()
print(f'Default params (NB21 config): {sum(p.numel() for p in _m.parameters()):,}')
_m = HybridGRUFC(gru_hidden=256, fc_hidden=128, use_attention=True)
print(f'Large config with attention : {sum(p.numel() for p in _m.parameters()):,}')
del _m

In [ ]:
class F1HybridDataset(Dataset):
    def __init__(self, windows, masks, strat, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.strat    = torch.tensor(strat,   dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = (None if targets is None
                         else torch.tensor(targets, dtype=torch.float32))

    def __len__(self): return len(self.windows)

    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], strat=self.strat[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None:
            d['target'] = self.targets[i]
        return d


def make_loader(windows, masks, strat, cat_idxs, targets=None,
                shuffle=False, batch_size=FINAL_BATCH_SIZE):
    ds = F1HybridDataset(windows, masks, strat, cat_idxs, targets)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_logits = 0.0, []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            win  = batch['window'].to(device)
            mask = batch['mask'].to(device)
            st   = batch['strat'].to(device)
            drv  = batch['driver'].to(device)
            rc   = batch['race'].to(device)
            cmp  = batch['compound'].to(device)
            yr   = batch['year'].to(device)

            logits = model(win, mask, st, drv, rc, cmp, yr)

            if is_train:
                tgt  = batch['target'].to(device)
                loss = criterion(logits, tgt)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(win)

            all_logits.append(logits.detach().cpu().float())

    probs = torch.sigmoid(torch.cat(all_logits)).numpy()
    return probs, total_loss / max(len(loader.dataset), 1)


def train_fold(params, fold_id, device, max_epochs, patience, batch_size):
    """Train one fold with given params. Returns (oof_probs, best_auc, best_epoch)."""
    ws = params['window_size']

    tr_idx = np.where(TRAIN_FOLDS != fold_id)[0]
    va_idx = np.where(TRAIN_FOLDS == fold_id)[0]

    strat_sc  = StandardScaler()
    tr_strat  = strat_sc.fit_transform(TRAIN_STRAT_RAW[tr_idx])
    va_strat  = strat_sc.transform(TRAIN_STRAT_RAW[va_idx])

    tr_cat = {c: TRAIN_CAT[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: TRAIN_CAT[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(TRAIN_WINDOWS[ws][tr_idx], TRAIN_MASKS[ws][tr_idx],
                            tr_strat, tr_cat, TRAIN_TARGETS[tr_idx],
                            shuffle=True, batch_size=batch_size)
    va_loader = make_loader(TRAIN_WINDOWS[ws][va_idx], TRAIN_MASKS[ws][va_idx],
                            va_strat, va_cat, batch_size=batch_size)

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([POS_WEIGHT], device=device)
    )
    model = HybridGRUFC(
        gru_hidden    = params['gru_hidden'],
        fc_hidden     = params['fc_hidden'],
        gru_drop      = params['gru_dropout'],
        head_drop1    = params['head_dropout1'],
        head_drop2    = params['head_dropout2'],
        use_attention = params['use_attention'],
    ).to(device)

    # DataParallel for final training (skipped during Optuna to keep per-GPU isolation)
    if device.type == 'cuda' and torch.cuda.device_count() > 1 and not params.get('_optuna_trial', False):
        model = nn.DataParallel(model)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params['lr'], weight_decay=params['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_auc, best_epoch, patience_ctr, best_state = 0.0, 0, 0, None

    for epoch in range(max_epochs):
        _, _ = run_epoch(model, tr_loader, criterion, optimizer, device)
        scheduler.step()
        val_probs, _ = run_epoch(model, va_loader, criterion, device=device)
        val_auc = roc_auc_score(TRAIN_TARGETS[va_idx], val_probs)

        if val_auc > best_auc:
            best_auc, best_epoch, patience_ctr = val_auc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in
                          (model.module if hasattr(model, 'module') else model).state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= patience:
            break

    (model.module if hasattr(model, 'module') else model).load_state_dict(
        {k: v.to(device) for k, v in best_state.items()}
    )
    return model, best_state, best_auc, best_epoch

## Optuna Objective — 2-Fold Evaluation (Folds 0 + 3)

In [ ]:
def objective(trial):
    params = {
        'window_size'  : trial.suggest_categorical('window_size',   [5, 10, 15, 20]),
        'gru_hidden'   : trial.suggest_categorical('gru_hidden',    [64, 128, 256]),
        'fc_hidden'    : trial.suggest_categorical('fc_hidden',     [32, 64, 128]),
        'gru_dropout'  : trial.suggest_float('gru_dropout',         0.05, 0.30),
        'head_dropout1': trial.suggest_float('head_dropout1',       0.10, 0.40),
        'head_dropout2': trial.suggest_float('head_dropout2',       0.05, 0.20),
        'lr'           : trial.suggest_float('lr',                  1e-4, 1e-3, log=True),
        'weight_decay' : trial.suggest_float('weight_decay',        1e-5, 1e-3, log=True),
        'use_attention': trial.suggest_categorical('use_attention', [True, False]),
        '_optuna_trial': True,  # suppresses DataParallel inside train_fold
    }

    fold_aucs = []
    for fold_id in TUNING_FOLDS:
        try:
            _, _, auc, _ = train_fold(
                params, fold_id,
                device     = DEVICE,
                max_epochs = TUNE_MAX_EPOCHS,
                patience   = TUNE_PATIENCE,
                batch_size = TUNE_BATCH_SIZE,
            )
            fold_aucs.append(auc)
        except Exception as e:
            print(f'  Trial {trial.number} fold {fold_id} error: {e}')
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(fold_aucs))

## Optuna Tuning — T4×2 Parallel Workers

On Kaggle T4×2: each `Process` pins to `cuda:0` or `cuda:1`.
On local CPU: falls back to single-process sequential tuning.

In [ ]:
WORKER_SCRIPT = '''import sys, argparse, warnings, pickle
import numpy as np
import torch
import torch.nn as nn
import optuna
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

parser = argparse.ArgumentParser()
parser.add_argument('--gpu',        type=int, required=True)
parser.add_argument('--n-trials',   type=int, required=True)
parser.add_argument('--study-name', type=str, required=True)
parser.add_argument('--storage',    type=str, required=True)
parser.add_argument('--data-dir',   type=str, required=True)
args = parser.parse_args()

device = torch.device(f'cuda:{args.gpu}' if torch.cuda.is_available() else 'cpu')
print(f'Worker gpu={args.gpu}: device={device}', flush=True)

# ── Constants ─────────────────────────────────────────────────────────
WINDOW_CANDIDATES = [5, 10, 15, 20]
CAT_COLS   = ['Driver', 'Race', 'Compound', 'Year']
STRAT_COLS = [
    'Stint', 'RaceProgress', 'laps_remaining', 'compound_ordinal', 'is_wet_tyre',
    'prime_pit_window', 'laps_to_driver_end', 'abs_position_change',
    'pos_change_rolling_std_3', 'PitStop_lag1',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TUNING_FOLDS    = [0, 3]
TUNE_MAX_EPOCHS = 50
TUNE_PATIENCE   = 7
TUNE_BATCH_SIZE = 2048
POS_WEIGHT      = 4.03

# ── Load data ─────────────────────────────────────────────────────────
d = Path(args.data_dir)
TRAIN_WINDOWS = {ws: np.load(d / f'train_win_{ws}.npy')  for ws in WINDOW_CANDIDATES}
TRAIN_MASKS   = {ws: np.load(d / f'train_mask_{ws}.npy') for ws in WINDOW_CANDIDATES}
TRAIN_STRAT_RAW = np.load(d / 'train_strat.npy')
TRAIN_TARGETS   = np.load(d / 'train_targets.npy')
TRAIN_FOLDS     = np.load(d / 'train_folds.npy')
with open(d / 'train_cat.pkl', 'rb') as f: TRAIN_CAT = pickle.load(f)
with open(d / 'emb_dims.pkl',  'rb') as f: EMB_DIMS  = pickle.load(f)
print(f'Data loaded. Train rows: {len(TRAIN_TARGETS):,}', flush=True)

# ── Model ─────────────────────────────────────────────────────────────
class HybridGRUFC(nn.Module):
    def __init__(self, n_strat=13, n_seq=5, gru_hidden=128, fc_hidden=64,
                 n_layers=2, gru_drop=0.15, head_drop1=0.25, head_drop2=0.10,
                 use_attention=False, emb_dims=None):
        super().__init__()
        if emb_dims is None:
            emb_dims = EMB_DIMS
        self.use_attention = use_attention
        self.driver_emb   = nn.Embedding(*emb_dims['Driver'])
        self.race_emb     = nn.Embedding(*emb_dims['Race'])
        self.compound_emb = nn.Embedding(*emb_dims['Compound'])
        self.year_emb     = nn.Embedding(*emb_dims['Year'])
        self.gru = nn.GRU(input_size=n_seq, hidden_size=gru_hidden,
                          num_layers=n_layers, batch_first=True,
                          dropout=gru_drop if n_layers > 1 else 0.0)
        if use_attention:
            self.attn_proj = nn.Linear(gru_hidden, 1)
        self.strat_fc = nn.Sequential(
            nn.Linear(n_strat, fc_hidden), nn.BatchNorm1d(fc_hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(fc_hidden, fc_hidden), nn.BatchNorm1d(fc_hidden), nn.ReLU(),
        )
        emb_total = sum(v[1] for v in emb_dims.values())
        merge_dim = gru_hidden + fc_hidden + emb_total
        self.head = nn.Sequential(
            nn.Linear(merge_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(head_drop1),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(head_drop2),
            nn.Linear(64, 1),
        )

    def forward(self, window, mask, strat, driver, race, compound, year):
        emb = torch.cat([self.driver_emb(driver), self.race_emb(race),
                         self.compound_emb(compound), self.year_emb(year)], dim=1)
        seq_len = mask.sum(dim=1).long().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            window, seq_len.cpu(), batch_first=True, enforce_sorted=False)
        if self.use_attention:
            gru_out_packed, _ = self.gru(packed)
            gru_out, _ = nn.utils.rnn.pad_packed_sequence(
                gru_out_packed, batch_first=True, total_length=window.size(1))
            scores = self.attn_proj(gru_out).masked_fill(~mask.unsqueeze(-1), -1e9)
            gru_feat = (torch.softmax(scores, dim=1) * gru_out).sum(dim=1)
        else:
            _, h_n   = self.gru(packed)
            gru_feat = h_n[-1]
        merged = torch.cat([gru_feat, self.strat_fc(strat), emb], dim=1)
        return self.head(merged).squeeze(1)


# ── Dataset + loader ──────────────────────────────────────────────────
class F1HybridDataset(Dataset):
    def __init__(self, windows, masks, strat, cat_idxs, targets=None):
        self.windows  = torch.tensor(windows, dtype=torch.float32)
        self.masks    = torch.tensor(masks,   dtype=torch.bool)
        self.strat    = torch.tensor(strat,   dtype=torch.float32)
        self.driver   = torch.tensor(cat_idxs['Driver'],   dtype=torch.long)
        self.race     = torch.tensor(cat_idxs['Race'],     dtype=torch.long)
        self.compound = torch.tensor(cat_idxs['Compound'], dtype=torch.long)
        self.year     = torch.tensor(cat_idxs['Year'],     dtype=torch.long)
        self.targets  = None if targets is None else torch.tensor(targets, dtype=torch.float32)
    def __len__(self): return len(self.windows)
    def __getitem__(self, i):
        d = dict(window=self.windows[i], mask=self.masks[i], strat=self.strat[i],
                 driver=self.driver[i], race=self.race[i],
                 compound=self.compound[i], year=self.year[i])
        if self.targets is not None: d['target'] = self.targets[i]
        return d

def make_loader(windows, masks, strat, cat_idxs, targets=None, shuffle=False):
    ds = F1HybridDataset(windows, masks, strat, cat_idxs, targets)
    return DataLoader(ds, batch_size=TUNE_BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(device.type == 'cuda'))


# ── Training ──────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None):
    model.train(optimizer is not None)
    all_logits = []
    ctx = torch.enable_grad() if optimizer is not None else torch.no_grad()
    with ctx:
        for batch in loader:
            win = batch['window'].to(device); mask = batch['mask'].to(device)
            st  = batch['strat'].to(device);  drv  = batch['driver'].to(device)
            rc  = batch['race'].to(device);   cmp  = batch['compound'].to(device)
            yr  = batch['year'].to(device)
            logits = model(win, mask, st, drv, rc, cmp, yr)
            if optimizer is not None:
                loss = criterion(logits, batch['target'].to(device))
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            all_logits.append(logits.detach().cpu().float())
    return torch.sigmoid(torch.cat(all_logits)).numpy()

def train_fold(params, fold_id):
    ws     = params['window_size']
    tr_idx = np.where(TRAIN_FOLDS != fold_id)[0]
    va_idx = np.where(TRAIN_FOLDS == fold_id)[0]
    sc = StandardScaler()
    tr_strat = sc.fit_transform(TRAIN_STRAT_RAW[tr_idx])
    va_strat = sc.transform(TRAIN_STRAT_RAW[va_idx])
    tr_cat = {c: TRAIN_CAT[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: TRAIN_CAT[c][va_idx] for c in CAT_COLS}
    tr_loader = make_loader(TRAIN_WINDOWS[ws][tr_idx], TRAIN_MASKS[ws][tr_idx],
                            tr_strat, tr_cat, TRAIN_TARGETS[tr_idx], shuffle=True)
    va_loader = make_loader(TRAIN_WINDOWS[ws][va_idx], TRAIN_MASKS[ws][va_idx],
                            va_strat, va_cat)
    crit  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=device))
    model = HybridGRUFC(gru_hidden=params['gru_hidden'], fc_hidden=params['fc_hidden'],
                        gru_drop=params['gru_dropout'], head_drop1=params['head_dropout1'],
                        head_drop2=params['head_dropout2'],
                        use_attention=params['use_attention']).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
    best_auc, patience_ctr, best_state = 0.0, 0, None
    for epoch in range(TUNE_MAX_EPOCHS):
        run_epoch(model, tr_loader, crit, opt); sched.step()
        val_probs = run_epoch(model, va_loader, crit)
        auc = roc_auc_score(TRAIN_TARGETS[va_idx], val_probs)
        if auc > best_auc:
            best_auc, patience_ctr = auc, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= TUNE_PATIENCE:
            break
    return best_auc


# ── Objective ─────────────────────────────────────────────────────────
def objective(trial):
    params = {
        'window_size'  : trial.suggest_categorical('window_size',   [5, 10, 15, 20]),
        'gru_hidden'   : trial.suggest_categorical('gru_hidden',    [64, 128, 256]),
        'fc_hidden'    : trial.suggest_categorical('fc_hidden',     [32, 64, 128]),
        'gru_dropout'  : trial.suggest_float('gru_dropout',         0.05, 0.30),
        'head_dropout1': trial.suggest_float('head_dropout1',       0.10, 0.40),
        'head_dropout2': trial.suggest_float('head_dropout2',       0.05, 0.20),
        'lr'           : trial.suggest_float('lr',                  1e-4, 1e-3, log=True),
        'weight_decay' : trial.suggest_float('weight_decay',        1e-5, 1e-3, log=True),
        'use_attention': trial.suggest_categorical('use_attention', [True, False]),
    }
    fold_aucs = []
    for fold_id in TUNING_FOLDS:
        try:
            auc = train_fold(params, fold_id)
            fold_aucs.append(auc)
        except Exception as e:
            print(f'  Trial {trial.number} fold {fold_id} error: {e}', flush=True)
            raise optuna.exceptions.TrialPruned()
    mean_auc = float(np.mean(fold_aucs))
    print(f'  Trial {trial.number}: {mean_auc:.4f}  params={params}', flush=True)
    return mean_auc


# ── Run ───────────────────────────────────────────────────────────────
study = optuna.load_study(study_name=args.study_name, storage=args.storage)
study.optimize(objective, n_trials=args.n_trials, show_progress_bar=False)
print(f'Worker gpu={args.gpu} done. Best: {study.best_value:.4f}', flush=True)
'''

In [ ]:
import subprocess, sys, os

# Write the self-contained worker script to disk (avoids CUDA fork/pickle issues)
worker_path = working_dir / 'worker_nb23.py'
worker_path.write_text(WORKER_SCRIPT, encoding='utf-8')
print(f'Worker script written to {worker_path}')

# Create shared SQLite-backed study
study = optuna.create_study(
    direction='maximize', study_name=STUDY_NAME,
    storage=STORAGE_URL, load_if_exists=True,
)
n_existing = len(study.trials)
print(f'Existing trials: {n_existing}')
n_remaining = max(0, N_TRIALS_PER_WORKER - n_existing // 2)

common_args = [
    '--study-name', STUDY_NAME,
    '--storage',    STORAGE_URL,
    '--data-dir',   str(working_dir),
    '--n-trials',   str(n_remaining),
]

if n_remaining == 0:
    print('Study already complete — skipping tuning.')
elif torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print(f'Launching 2 subprocess workers x {n_remaining} trials each...')
    p0 = subprocess.Popen(
        [sys.executable, str(worker_path), '--gpu', '0'] + common_args,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    p1 = subprocess.Popen(
        [sys.executable, str(worker_path), '--gpu', '1'] + common_args,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    # Stream output from both workers
    import threading
    def stream(proc, label):
        for line in proc.stdout:
            print(f'[{label}] {line}', end='', flush=True)
    t0 = threading.Thread(target=stream, args=(p0, 'gpu0'), daemon=True)
    t1 = threading.Thread(target=stream, args=(p1, 'gpu1'), daemon=True)
    t0.start(); t1.start()
    p0.wait(); p1.wait()
    t0.join();  t1.join()
    print(f'Both workers done. Exit codes: {p0.returncode}, {p1.returncode}')
else:
    # Single-process fallback
    total = n_remaining * 2
    print(f'Single worker: {total} trials on {DEVICE}...')
    p = subprocess.Popen(
        [sys.executable, str(worker_path), '--gpu', '0'] + ['--n-trials', str(total)] + common_args[2:],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()

study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE_URL)
print(f'\nTotal completed trials: {len(study.trials)}')
print(f'Best 2-fold AUC: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')

## Study Analysis — Best Params and Top Trials

In [ ]:
study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE_URL)

df_trials = study.trials_dataframe()
completed = df_trials[df_trials['state'] == 'COMPLETE'].sort_values('value', ascending=False)
print(f'Completed trials: {len(completed)}')
print('\nTop 10 trials:')
param_cols = [c for c in completed.columns if c.startswith('params_')]
show_cols  = ['number', 'value'] + param_cols
print(completed[show_cols].head(10).to_string(index=False))

BEST_PARAMS = study.best_params.copy()
BEST_PARAMS['_optuna_trial'] = False   # allow DataParallel in final run
print(f'\nBest params saved to BEST_PARAMS:')
for k, v in BEST_PARAMS.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

# Projected AUC using best params
print(f'\nBest 2-fold CV AUC  : {study.best_value:.4f}')
nb21_solo_auc = 0.9185
print(f'NB21 2-fold-equivalent AUC: ~{nb21_solo_auc:.4f} (full 5-fold reference)')
print(f'Gain estimate: {study.best_value - nb21_solo_auc:+.4f}')

## Full 5-Fold Training with Best Params

In [ ]:
oof_preds        = np.zeros(len(TRAIN_IDS))
test_preds_folds = []
fold_aucs        = []
ws               = BEST_PARAMS['window_size']

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
)

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'Fold {fold}  (W={ws}, gru={BEST_PARAMS["gru_hidden"]}, fc={BEST_PARAMS["fc_hidden"]}, attn={BEST_PARAMS["use_attention"]})')
    print('='*60)

    model, best_state, best_auc, best_epoch = train_fold(
        BEST_PARAMS, fold,
        device     = DEVICE,
        max_epochs = FINAL_MAX_EPOCHS,
        patience   = FINAL_PATIENCE,
        batch_size = FINAL_BATCH_SIZE,
    )

    # OOF predictions
    va_idx = np.where(TRAIN_FOLDS == fold)[0]
    strat_sc = StandardScaler()
    strat_sc.fit(TRAIN_STRAT_RAW[np.where(TRAIN_FOLDS != fold)[0]])
    va_strat = strat_sc.transform(TRAIN_STRAT_RAW[va_idx])
    va_cat   = {c: TRAIN_CAT[c][va_idx] for c in CAT_COLS}
    va_loader = make_loader(TRAIN_WINDOWS[ws][va_idx], TRAIN_MASKS[ws][va_idx],
                            va_strat, va_cat, batch_size=FINAL_BATCH_SIZE)

    oof_probs, _ = run_epoch(model, va_loader, criterion)
    oof_preds[va_idx] = oof_probs
    fold_auc = roc_auc_score(TRAIN_TARGETS[va_idx], oof_probs)
    fold_aucs.append(fold_auc)

    # Test predictions
    te_strat  = strat_sc.transform(TEST_STRAT_RAW)
    te_loader = make_loader(TEST_WINDOWS[ws], TEST_MASKS[ws],
                            te_strat, TEST_CAT, batch_size=FINAL_BATCH_SIZE)
    te_probs, _ = run_epoch(model, te_loader, criterion)
    test_preds_folds.append(te_probs)

    torch.save(best_state, models_dir / f'hybrid_nb23_fold{fold}.pt')
    print(f'  Fold {fold} AUC: {fold_auc:.4f}  best_epoch: {best_epoch+1}')

oof_auc_main = roc_auc_score(TRAIN_TARGETS, oof_preds)
test_preds_main = np.mean(np.stack(test_preds_folds, axis=0), axis=0)

print(f'\nOOF AUC  : {oof_auc_main:.4f}')
print(f'Fold AUCs: {[f"{a:.4f}" for a in fold_aucs]}')
print(f'Fold std : {np.std(fold_aucs):.4f}')
print(f'NB21 ref : 0.9185')
print(f'Gain     : {oof_auc_main - 0.9185:+.4f}')

## SWA Refinement (Stochastic Weight Averaging)

Continues training from each fold's converged weights for `SWA_EPOCHS` more epochs with a flat low LR, then averages the weights. SWA produces a wider minimum → better generalisation on unseen races.

In [ ]:
from torch.optim.swa_utils import AveragedModel, SWALR

oof_preds_swa    = np.zeros(len(TRAIN_IDS))
test_preds_swa   = []
fold_aucs_swa    = []
ws               = BEST_PARAMS['window_size']

def update_bn_dict(loader, model, device):
    """update_bn replacement for dict-yielding DataLoaders.
    PyTorch's update_bn calls input.to(device) on the first batch element,
    which fails when the loader yields dicts instead of tensors.
    """
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.num_batches_tracked.zero_()
            m.running_mean.zero_()
            m.running_var.fill_(1)
    model.train()
    with torch.no_grad():
        for batch in loader:
            model(batch['window'].to(device), batch['mask'].to(device),
                  batch['strat'].to(device),  batch['driver'].to(device),
                  batch['race'].to(device),   batch['compound'].to(device),
                  batch['year'].to(device))
    model.eval()

for fold in range(N_FOLDS):
    print(f'Fold {fold} SWA...', end=' ', flush=True)

    tr_idx = np.where(TRAIN_FOLDS != fold)[0]
    va_idx = np.where(TRAIN_FOLDS == fold)[0]

    strat_sc = StandardScaler()
    tr_strat = strat_sc.fit_transform(TRAIN_STRAT_RAW[tr_idx])
    va_strat = strat_sc.transform(TRAIN_STRAT_RAW[va_idx])
    te_strat = strat_sc.transform(TEST_STRAT_RAW)

    tr_cat = {c: TRAIN_CAT[c][tr_idx] for c in CAT_COLS}
    va_cat = {c: TRAIN_CAT[c][va_idx] for c in CAT_COLS}

    tr_loader = make_loader(TRAIN_WINDOWS[ws][tr_idx], TRAIN_MASKS[ws][tr_idx],
                            tr_strat, tr_cat, TRAIN_TARGETS[tr_idx],
                            shuffle=True, batch_size=FINAL_BATCH_SIZE)
    va_loader = make_loader(TRAIN_WINDOWS[ws][va_idx], TRAIN_MASKS[ws][va_idx],
                            va_strat, va_cat, batch_size=FINAL_BATCH_SIZE)
    te_loader = make_loader(TEST_WINDOWS[ws], TEST_MASKS[ws],
                            te_strat, TEST_CAT, batch_size=FINAL_BATCH_SIZE)

    # Load converged weights from standard training
    base_model = HybridGRUFC(
        gru_hidden    = BEST_PARAMS['gru_hidden'],
        fc_hidden     = BEST_PARAMS['fc_hidden'],
        gru_drop      = BEST_PARAMS['gru_dropout'],
        head_drop1    = BEST_PARAMS['head_dropout1'],
        head_drop2    = BEST_PARAMS['head_dropout2'],
        use_attention = BEST_PARAMS['use_attention'],
    ).to(DEVICE)
    base_model.load_state_dict(
        torch.load(models_dir / f'hybrid_nb23_fold{fold}.pt', map_location=DEVICE)
    )

    swa_model = AveragedModel(base_model)
    swa_scheduler = SWALR(
        torch.optim.AdamW(base_model.parameters(), lr=SWA_LR),
        swa_lr=SWA_LR, anneal_epochs=SWA_EPOCHS,
    )
    criterion_swa = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE)
    )

    for _ in range(SWA_EPOCHS):
        run_epoch(base_model, tr_loader, criterion_swa,
                  optimizer=torch.optim.AdamW(base_model.parameters(), lr=SWA_LR),
                  device=DEVICE)
        swa_model.update_parameters(base_model)
        swa_scheduler.step()

    # Update BN stats for SWA model (custom — handles dict batches)
    update_bn_dict(tr_loader, swa_model, DEVICE)

    # Evaluate SWA model
    oof_probs_swa, _ = run_epoch(swa_model, va_loader, criterion_swa, device=DEVICE)
    oof_preds_swa[va_idx] = oof_probs_swa
    fold_auc_swa = roc_auc_score(TRAIN_TARGETS[va_idx], oof_probs_swa)
    fold_aucs_swa.append(fold_auc_swa)

    te_probs_swa, _ = run_epoch(swa_model, te_loader, criterion_swa, device=DEVICE)
    test_preds_swa.append(te_probs_swa)

    print(f'AUC={fold_auc_swa:.4f}')

oof_auc_swa = roc_auc_score(TRAIN_TARGETS, oof_preds_swa)
test_preds_swa_mean = np.mean(np.stack(test_preds_swa, axis=0), axis=0)

print(f'\nSWA OOF AUC  : {oof_auc_swa:.4f}  (std={np.std(fold_aucs_swa):.4f})')
print(f'Main OOF AUC : {oof_auc_main:.4f}')
print(f'SWA gain     : {oof_auc_swa - oof_auc_main:+.4f}')


## Choose Best Variant + Correlation Analysis

In [ ]:
# Use SWA if it improves by > 0.0005; otherwise use standard
USE_SWA = (oof_auc_swa - oof_auc_main) > 0.0005
print(f'SWA gain: {oof_auc_swa - oof_auc_main:+.4f}  -> using {"SWA" if USE_SWA else "standard"} predictions')

final_oof_preds  = oof_preds_swa     if USE_SWA else oof_preds
final_test_preds = test_preds_swa_mean if USE_SWA else test_preds_main
final_oof_auc    = oof_auc_swa       if USE_SWA else oof_auc_main

print(f'\nFinal OOF AUC: {final_oof_auc:.4f}')

# Load reference OOF predictions for correlation analysis
lgbm_df    = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
nb21_df    = pd.read_parquet(processed_dir / 'oof_predictions_nb21.parquet')

lgbm_vals  = lgbm_df.set_index('id').loc[TRAIN_IDS, 'lgbm_tuned_oof'].values
nb21_vals  = nb21_df.set_index('id').loc[TRAIN_IDS, 'hybrid_oof'].values

rho_vs_lgbm, _ = spearmanr(final_oof_preds, lgbm_vals)
rho_vs_nb21, _ = spearmanr(final_oof_preds, nb21_vals)

print(f'Spearman rho vs LGBM NB12: {rho_vs_lgbm:.4f}')
print(f'Spearman rho vs Hybrid NB21: {rho_vs_nb21:.4f}')

# NB21 seed ensemble preview (rank avg NB21_s0 + NB21_s1 + NB21_s2 + NB23)
try:
    s1 = pd.read_parquet(processed_dir / 'oof_predictions_nb21_seed1.parquet')
    s2 = pd.read_parquet(processed_dir / 'oof_predictions_nb21_seed2.parquet')
    nb21_s0 = nb21_vals
    nb21_s1 = s1.set_index('id').loc[TRAIN_IDS, 'hybrid_oof'].values
    nb21_s2 = s2.set_index('id').loc[TRAIN_IDS, 'hybrid_oof'].values

    from scipy.stats import rankdata as rk
    n = len(nb21_s0)
    rank_3seed = (rk(nb21_s0)/n + rk(nb21_s1)/n + rk(nb21_s2)/n) / 3
    rank_4seed = (rk(nb21_s0)/n + rk(nb21_s1)/n + rk(nb21_s2)/n + rk(final_oof_preds)/n) / 4

    auc_3seed = roc_auc_score(TRAIN_TARGETS, rank_3seed)
    auc_4seed = roc_auc_score(TRAIN_TARGETS, rank_4seed)
    print(f'\nEnsemble previews (rank avg):')
    print(f'  NB21 3-seed (reference)          : {auc_3seed:.4f}')
    print(f'  NB21 3-seed + NB23 (4-model avg) : {auc_4seed:.4f}  delta={auc_4seed-auc_3seed:+.4f}')
    if auc_4seed - auc_3seed > 0.001:
        print('  -> NB23 adds meaningful signal to 3-seed ensemble')
    else:
        print('  -> NB23 correlated with NB21 seeds — submit NB23 solo only')
except FileNotFoundError:
    print('NB21 seed parquets not found — skipping ensemble preview')

BOOST_SOLO      = 0.0177
BOOST_MULTISEED = 0.0142
print(f'\nProjected LB (NB23 solo, +{BOOST_SOLO}): {final_oof_auc + BOOST_SOLO:.4f}')
print(f'Current best LB: 0.93651  Gap: {final_oof_auc + BOOST_SOLO - 0.93651:+.4f}')

## Save OOF + Test Prediction Parquets

In [ ]:
oof_df = pd.DataFrame({
    'id':         TRAIN_IDS,
    'fold':       TRAIN_FOLDS,
    'PitNextLap': TRAIN_TARGETS.astype(int),
    'hybrid_oof': final_oof_preds,
})
assert oof_df['id'].nunique() == len(oof_df), 'Duplicate ids in OOF'
oof_df.to_parquet(processed_dir / 'oof_predictions_nb23.parquet', index=False)
print(f'Saved oof_predictions_nb23.parquet  ({len(oof_df):,} rows)')

test_df = pd.DataFrame({'id': TEST_IDS, 'hybrid_pred': final_test_preds})
test_df.to_parquet(processed_dir / 'test_predictions_nb23.parquet', index=False)
print(f'Saved test_predictions_nb23.parquet ({len(test_df):,} rows)')

summary = {
    'oof_auc':        final_oof_auc,
    'fold_aucs':      fold_aucs_swa if USE_SWA else fold_aucs,
    'fold_std':       float(np.std(fold_aucs_swa if USE_SWA else fold_aucs)),
    'oof_auc_main':   oof_auc_main,
    'oof_auc_swa':    oof_auc_swa,
    'used_swa':       USE_SWA,
    'best_params':    {k:v for k,v in BEST_PARAMS.items() if not k.startswith('_')},
    'rho_vs_lgbm':    float(rho_vs_lgbm),
    'rho_vs_nb21':    float(rho_vs_nb21),
    'nb21_ref_auc':   0.9185,
    'gain_vs_nb21':   float(final_oof_auc - 0.9185),
}
with open(models_dir / 'nb23_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)
print('Saved nb23_summary.pkl')
for k, v in summary.items():
    if k != 'best_params':
        print(f'  {k}: {v}')

## Submission CSV

In [ ]:
if Path('/kaggle/input').exists():
    submissions_dir = Path('/kaggle/working')
else:
    cwd = Path.cwd()
    while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
        cwd = cwd.parent
    submissions_dir = cwd / 'submissions'

sub = test_df[['id']].copy()
sub['PitNextLap'] = final_test_preds
sub = sub.sort_values('id').reset_index(drop=True)

assert sub['id'].duplicated().sum() == 0,      'Duplicate IDs!'
assert sub['PitNextLap'].isnull().sum() == 0,  'NaN predictions!'

out_path = submissions_dir / 'submission_v011_hybrid_nb23.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {sub.shape}')
print(f'PitNextLap range: [{sub.PitNextLap.min():.4f}, {sub.PitNextLap.max():.4f}]')
print(f'\nOOF AUC  : {final_oof_auc:.4f}')
print(f'Proj LB  : {final_oof_auc + BOOST_SOLO:.4f}  (using +{BOOST_SOLO} solo boost)')
print(f'Best LB  : 0.93651')
print(f'Gap      : {final_oof_auc + BOOST_SOLO - 0.93651:+.4f}')